# Uniform Manifold Approximation and Projection (UMAP) — From Theory to Practice

## What This Notebook Covers
This notebook is the practical, interactive companion to the **UMAP README**. We will explore how high-dimensional, non-linear manifolds can be compressed and visualized in a low-dimensional space. You will learn to load high-dimensional digit image datasets, standard-scale and normalize inputs, apply PCA as an initial noise-filtering step, implement a simplified UMAP-like force-directed layout solver from scratch using only NumPy, run optimized production-grade Numba-compiled `umap-learn` pipelines, tune critical hyperparameters like `n_neighbors` and `min_dist`, compare UMAP side-by-side against PCA and t-SNE, and quantitatively evaluate embedding quality using the Trustworthiness metric and downstream KNN accuracy sweeps.

## What You Will Accomplish
- Describe the principles of manifold learning and the mathematical difference between UMAP's cross-entropy loss and t-SNE's KL divergence.
- Prepare high-dimensional image data (Standardization and PCA preprocessing) before manifold projection.
- Build a simplified, force-directed neighbor embedding solver from scratch using pure NumPy matrix multiplication to compute attractive and repulsive forces.
- Apply the official `umap-learn` library to compress 5,000 samples of handwritten digit images from 784 dimensions to 2 dimensions.
- Visualize the resulting embeddings using scatter plots and contour density distributions to assess cluster separation.
- Run hyperparameter sweeps over `n_neighbors` and `min_dist` to observe the transition from fragmented local structures to merged global configurations.
- Compare PCA, t-SNE, and UMAP side-by-side on a shared dataset to understand their strengths and weaknesses.
- Evaluate embedding quality quantitatively using Scikit-learn's `trustworthiness` score and cross-validated KNN classifier accuracy.

## Before You Start (Prerequisites)
- Comfort manipulating Python loops, dictionaries, and NumPy array slices.
- Familiarity with basic linear algebra concepts (vectors, dot products, matrix multiplication `@`).
- Basic familiarity with PCA and t-SNE is helpful but not required.

## About the Dataset
We use the benchmark **MNIST Handwritten Digits dataset**, containing 70,000 grayscale images of handwritten digits (0–9), each of size 28×28 pixels. Each image is represented as a flat vector of **784 numbers** (pixel brightness values from 0 to 255). Our goal is to compress these 784 dimensions down to just 2 dimensions, keeping similar digits close together.

We load it using `sklearn.datasets.fetch_openml`.
---

## 1. Setup & Workspace Preparation

### WHY?
Importing all scientific computing, visualization, and validation libraries at the beginning of the notebook prevents path resolution errors and ensures all seeds are fixed for reproducibility.

### HOW?
We import NumPy, Pandas, Matplotlib, Seaborn, and key Scikit-learn modules, set formatting options, and fix numpy seed to 42.

In [ ]:
# Import NumPy for manual matrix operations and distance computations
import numpy as np

# Import Pandas for displaying dataframes and statistical summaries
import pandas as pd

# Import visualization libraries for plotting scatter and contour charts
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

# Import dataset loader, preprocessors, and dimensionality reduction tools from sklearn
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, classification_report

# Try importing production UMAP library, install it if missing
try:
    import umap
except ImportError:
    import subprocess
    import sys
    print("Installing umap-learn...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "umap-learn"])
    import umap

# Set utilities for timing code execution and suppressing warnings
import time
import warnings
warnings.filterwarnings('ignore')

# Set seaborn style for clean grids
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

# Fix numpy seed for reproducibility
np.random.seed(42)

print("All libraries imported and seed fixed to 42.")

## 2. Dataset Loading & Stratified Subsampling

### WHY?
UMAP runs close to linear time, but performing distance metrics on 70,000 pixels is still computationally expensive. We take a stratified subsample of 5,000 samples (500 per digit class) to ensure balanced representation while keeping execution times short.

### HOW?
We load MNIST using `fetch_openml`, select the subset proportionally using index slicing, and verify the class distribution using value counts.

In [ ]:
print("📥 Loading MNIST dataset from OpenML (this may take up to 60 seconds)...\n")

# Fetch MNIST dataset (as_frame=False to return numpy arrays directly)
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_raw = mnist.data.astype(np.float32)
y_raw = mnist.target.astype(int)

print(f"✅ Full dataset loaded! Shape: {X_raw.shape[0]} samples × {X_raw.shape[1]} features")

# Take a stratified subsample of 5,000 samples (500 per digit class)
samples_per_class = 500
selected_indices = []

for digit in range(10):
    digit_indices = np.where(y_raw == digit)[0]
    chosen_indices = np.random.choice(digit_indices, size=samples_per_class, replace=False)
    selected_indices.extend(chosen_indices)

selected_indices = np.array(selected_indices)
# Shuffle selected indices to remove ordering bias
np.random.shuffle(selected_indices)

X = X_raw[selected_indices]
y = y_raw[selected_indices]

print(f"📊 Subsampled dataset created! Target shape: {X.shape[0]} samples × {X.shape[1]} features")
print(f"   Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

## 3. Exploratory Data Analysis (EDA)

### WHY?
Before performing dimensionality reduction, it is crucial to visually inspect sample images to understand their raw features and make sure the data loaded correctly.

### HOW?
We plot a grid of 25 random handwritten digit images from our subset, with each image labeled and colored according to its true digit class.

In [ ]:
# Set up the figure grid
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
fig.suptitle("Sample MNIST Handwritten Digits from Our Subsample", fontsize=16, fontweight='bold', y=0.95)

colors = plt.cm.tab10(np.linspace(0, 1, 10))

for i, ax in enumerate(axes.flat):
    # Reshape the flat 784-D vector back into a 28x28 grayscale image grid
    digit_img = X[i].reshape(28, 28)
    true_label = y[i]
    
    # Plot image with coordinates off
    ax.imshow(digit_img, cmap='gray_r')
    ax.axis('off')
    
    # Add colored boundary box representing the digit class
    rect = plt.Rectangle((0,0), 27, 27, fill=False, color=colors[true_label], linewidth=3)
    ax.add_patch(rect)
    ax.set_title(f"Digit: {true_label}", color=colors[true_label], fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Preprocessing: Standardization and PCA Pre-processing

### WHY?
Dimensionality reduction models are highly sensitive to variable scales. Because pixel values range from 0 to 255, dividing by 255.0 normalizes them to $[0, 1]$, making distance calculations scale-invariant. 

We then run **PCA** to reduce the dimensionality from 784 dimensions to 50 dimensions. This pre-processing step filters out high-frequency noise, speeds up subsequent UMAP neighbor graph computations, and prevents local optimization traps.

### HOW?
We divide the data by 255.0, fit `PCA(n_components=50)` on the normalized features, and plot a Scree plot showing cumulative variance.

In [ ]:
# Step 4a: Normalize pixel features to [0, 1] range
X_normalized = X / 255.0

# Step 4b: Apply PCA to reduce dimensionality to 50 components
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_normalized)

print(f"Original feature space dimension: {X_normalized.shape[1]} (pixels)")
print(f"Reduced feature space dimension : {X_pca.shape[1]} (principal components)")
print(f"Total variance retained in 50 components: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Step 4c: Plot PCA Scree plot diagnostic dashboard
fig, ax = plt.subplots(figsize=(10, 5))
components = np.arange(1, 51)
ax.bar(components, pca.explained_variance_ratio_, alpha=0.7, color='#2196F3', label='Individual Variance')
ax.step(components, np.cumsum(pca.explained_variance_ratio_), where='mid', color='#FF9800', label='Cumulative Variance')
ax.axhline(y=0.85, color='r', linestyle='--', label='85% Variance Threshold')
ax.set_xlabel('Principal Component Index', fontsize=11)
ax.set_ylabel('Variance Explained Ratio', fontsize=11)
ax.set_title('PCA Scree Plot: Filtering Image Noise', fontsize=13, fontweight='bold')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 5. Mathematical Blueprint of UMAP (Review)

UMAP models similarity in the original high-dimensional space ($w_{j|i}$) using a **Gaussian-like distribution with a local offset**:

$$w_{j|i} = \exp\left( - \frac{\max(0, d(x_i, x_j) - \rho_i)}{\sigma_i} \right)$$

To make the metrics robust to local density changes and outliers, we calculate the symmetric fuzzy set union joint probability:

$$p_{ij} = p_{i|j} + p_{j|i} - p_{i|j} p_{j|i}$$

In the low-dimensional map, we compute similarity $q_{ij}$ using a Student t-distribution curve:

$$q_{ij} = \left( 1 + a \cdot \lVert y_i - y_j \rVert^{2b} \right)^{-1}$$

We minimize the difference between these two distributions using the **Fuzzy Cross-Entropy loss function**:

$$\mathcal{L} = \sum_{i \neq j} \left[ p_{ij} \log\left(\frac{p_{ij}}{q_{ij}}\right) + (1 - p_{ij}) \log\left(\frac{1 - p_{ij}}{1 - q_{ij}}\right) \right]$$

---

## 6. Manual UMAP Implementation (From Scratch using NumPy)

### WHY?
Implementing a simplified UMAP-like layout algorithm from scratch using only NumPy helps you understand how neighborhood graphs are built, and how attractive and repulsive forces are calculated to position points in a 2D layout.

### HOW?
We implement the `SimpleUMAP` class. It builds a $K$-nearest neighbor graph from Euclidean distances, symmetrizes it, initializes coordinates using standard normal noise, and runs a force-directed layout update loop where connected points pull together and disconnected negative samples push apart.

In [ ]:
class SimpleUMAP:
    def __init__(self, n_neighbors=15, n_components=2, n_epochs=200, learning_rate=1.0, random_state=42):
        self.n_neighbors = n_neighbors
        self.n_components = n_components
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.costs = []

    def _compute_pairwise_distances(self, X):
        # Compute squared Euclidean distances using: ||a-b||^2 = ||a||^2 + ||b||^2 - 2*(a.b)
        sum_sq = np.sum(X**2, axis=1)
        D = sum_sq[:, np.newaxis] + sum_sq[np.newaxis, :] - 2 * np.dot(X, X.T)
        D = np.maximum(D, 0.0) # Prevent negative values due to floating-point issues
        return D

    def _build_neighborhood_graph(self, X):
        n = X.shape[0]
        D = self._compute_pairwise_distances(X)
        
        # Identify the k-nearest neighbors for each point (excluding self)
        nearest_indices = np.argsort(D, axis=1)[:, 1:self.n_neighbors+1]
        
        # Create adjacency matrix
        adjacency = np.zeros((n, n))
        for i in range(n):
            # Compute local scaling parameters
            # rho is the distance to the single nearest neighbor
            rho = np.sqrt(D[i, nearest_indices[i, 0]])
            
            # For simplicity, we assume constant sigma bandwidth scale = 1.0
            sigma = 1.0
            
            # Compute asymmetric similarities
            for j in nearest_indices[i]:
                dist = np.sqrt(D[i, j])
                val = np.exp(-np.maximum(0.0, dist - rho) / sigma)
                adjacency[i, j] = val
                
        # Symmetrize the graph using the fuzzy set union formula: P_ij = P_i|j + P_j|i - P_i|j*P_j|i
        adjacency = adjacency + adjacency.T - adjacency * adjacency.T
        return adjacency

    def fit_transform(self, X):
        np.random.seed(self.random_state)
        n = X.shape[0]
        
        # Step 6a: Compute neighborhood graph
        P = self._build_neighborhood_graph(X)
        
        # Step 6b: Initialize low-dimensional coordinates using random noise
        Y = np.random.randn(n, self.n_components) * 1e-4
        
        # UMAP curve parameters for min_dist = 0.1
        a, b = 1.93, 0.79
        
        # Run Stochastic Gradient Descent layout loop
        for epoch in range(self.n_epochs):
            # Compute target pairwise distances in 2D space
            D_y = self._compute_pairwise_distances(Y)
            
            # Compute low-dimensional similarity Q_ij
            num = 1.0 / (1.0 + a * (D_y**b))
            np.fill_diagonal(num, 0.0)
            
            # Compute attractive and repulsive gradients manually
            grad = np.zeros_like(Y)
            for i in range(n):
                for j in range(n):
                    if i == j:
                        continue
                    diff = Y[i] - Y[j]
                    dist_sq = D_y[i, j]
                    
                    # Attraction force: active for actual neighbors (P_ij > 0)
                    if P[i, j] > 0:
                        # Attract along gradient
                        attr = P[i, j] * (b * a * (dist_sq**(b-1))) / (1.0 + a * (dist_sq**b))
                        grad[i] += attr * diff
                        
                    # Repulsion force: active for non-neighbors (P_ij = 0)
                    # Using negative sampling for speed: only evaluate if random check hits
                    elif np.random.rand() < 0.05:
                        rep = (1.0 - P[i, j]) * (b * a * (dist_sq**(b-1))) / ((1e-12 + dist_sq) * (1.0 + a * (dist_sq**b)))
                        grad[i] -= rep * diff
            
            # Update coordinates using gradient step
            Y -= self.learning_rate * grad
            
            # Shift coordinates around origin to prevent drift
            Y -= np.mean(Y, axis=0)
            
            # Log loss progress
            if epoch % 50 == 0:
                # Approximate cost calculation on a subset to avoid slow evaluations
                loss = np.sum(P * np.log(np.maximum(P, 1e-12) / np.maximum(num, 1e-12)))
                self.costs.append(loss)
                print(f"Epoch {epoch:4d} | Approximation Loss: {loss:.4f}")
                
        return Y

## 7. Running and Visualizing Our From-Scratch Solver

### WHY?
We run our from-scratch NumPy solver on a small subset of 300 samples to verify that it functions correctly and converges to a reasonable layout.

### HOW?
We slice the first 300 samples of our pre-processed dataset, run our `SimpleUMAP` solver, and plot the loss convergence curve side-by-side with the final 2D projection scatter plot.

In [ ]:
# Slice a small subset for scratch execution speed
n_scratch = 300
X_scratch = X_pca[:n_scratch]
y_scratch = y[:n_scratch]

print(f"🚀 Running From-Scratch SimpleUMAP on {n_scratch} samples...")
scratch_umap = SimpleUMAP(n_neighbors=15, n_components=2, n_epochs=200, learning_rate=0.1, random_state=42)

t0 = time.time()
Y_scratch = scratch_umap.fit_transform(X_scratch)
t1 = time.time()

print(f"✅ Scratch solver complete! Runtime: {t1 - t0:.2f} seconds.\n")

# Plot the diagnostic dashboard side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Loss convergence curve
axes[0].plot(np.arange(0, 200, 50), scratch_umap.costs, marker='o', color='#E91E63', linewidth=2)
axes[0].set_xlabel('Iteration Index', fontsize=11)
axes[0].set_ylabel('Approximation Cross-Entropy Cost', fontsize=11)
axes[0].set_title('Loss Convergence (Scratch UMAP)', fontsize=13, fontweight='bold')

# Right: 2D Projected Space scatter plot
scatter = axes[1].scatter(Y_scratch[:, 0], Y_scratch[:, 1], c=y_scratch, cmap='tab10', alpha=0.8, edgecolors='black', s=50)
legend = axes[1].legend(*scatter.legend_elements(), title="Digits")
axes[1].add_artist(legend)
axes[1].set_xlabel('Component 1 (PC1)', fontsize=11)
axes[1].set_ylabel('Component 2 (PC2)', fontsize=11)
axes[1].set_title('Scratch 2D UMAP Projection Space (300 Samples)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Library-Grade UMAP Using `umap-learn`

### WHY?
While our manual solver is educational, the production-grade `umap-learn` library uses **Numba-compiled JIT optimizations** and approximate nearest-neighbor algorithms (NN-Descent), running orders of magnitude faster on massive datasets.

### HOW?
We configure `umap.UMAP` with default parameters (`n_neighbors=15`, `min_dist=0.1`), fit it to our pre-processed dataset of 5,000 samples, and log execution metrics.

In [ ]:
print("🚀 Running Production UMAP (umap-learn) on 5,000 samples...")
t_start = time.time()

# Configure and fit the production-grade UMAP estimator
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    init='spectral',
    random_state=42
)
Y_lib = reducer.fit_transform(X_pca)
t_end = time.time()

print(f"\n✅ Production UMAP complete!")
print(f"   Total runtime     : {t_end - t_start:.2f} seconds")
print(f"   Output shape      : {Y_lib.shape}")

## 9. Visualizing UMAP Embeddings

### WHY?
Visualizing the 2D projection space helps evaluate whether the dataset's classes are separated cleanly. We will plot both a standard cluster scatter plot and a density contour plot to reveal the cluster density centers.

### HOW?
We plot a 2D scatter plot colored by digit class, and a Seaborn KDE joint density contour plot showing cluster centers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Left: 2D Projected Space scatter plot colored by digit class
scatter = axes[0].scatter(Y_lib[:, 0], Y_lib[:, 1], c=y, cmap='tab10', alpha=0.7, edgecolors='none', s=20)
legend = axes[0].legend(*scatter.legend_elements(), title="Digits", loc='best')
axes[0].add_artist(legend)
axes[0].set_xlabel('UMAP Component 1', fontsize=11)
axes[0].set_ylabel('UMAP Component 2', fontsize=11)
axes[0].set_title('MNIST Digit Clusters in 2D UMAP Projected Space', fontsize=13, fontweight='bold')

# Right: Joint Density contour plot showing cluster centers
sns.kdeplot(
    x=Y_lib[:, 0], y=Y_lib[:, 1], hue=y, palette='tab10', fill=True,
    alpha=0.4, levels=5, thresh=0.1, ax=axes[1]
)
axes[1].set_xlabel('UMAP Component 1', fontsize=11)
axes[1].set_ylabel('UMAP Component 2', fontsize=11)
axes[1].set_title('Cluster Density Contour Distribution (KDE)', fontsize=13, fontweight='bold')

plt.suptitle('MNIST UMAP 2D Projection Space Diagnostic Dashboard', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 10. Comparing PCA vs. t-SNE vs. UMAP

### WHY?
Comparing dimensionality reduction techniques side-by-side helps visualize how linear (PCA), local non-linear (t-SNE), and manifold-preserving non-linear (UMAP) methods model the same structural features.

### HOW?
We run Scikit-learn's `PCA` and `TSNE` algorithms alongside `UMAP`, compile their execution runtimes, and plot their projected 2D coordinates side-by-side.

In [ ]:
# Run PCA compression for comparison
t0 = time.time()
pca_comp = PCA(n_components=2, random_state=42)
Y_pca_2d = pca_comp.fit_transform(X_normalized)
t_pca = time.time() - t0

# Run t-SNE compression for comparison
t0 = time.time()
tsne_comp = TSNE(n_components=2, perplexity=40, random_state=42, n_jobs=-1)
Y_tsne_2d = tsne_comp.fit_transform(X_pca)
t_tsne = time.time() - t0

# Compile timings
print("📊 Execution Timing Comparison:")
print(f"   PCA Runtime   : {t_pca:.4f} seconds")
print(f"   t-SNE Runtime : {t_tsne:.4f} seconds")
print(f"   UMAP Runtime  : {t_end - t_start:.4f} seconds\n")

# Plot projections side-by-side
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

# Plot PCA
axes[0].scatter(Y_pca_2d[:, 0], Y_pca_2d[:, 1], c=y, cmap='tab10', alpha=0.5, s=10)
axes[0].set_title(f"PCA (Linear Projection) | Time: {t_pca:.2f}s", fontsize=14, fontweight='bold')
axes[0].set_xlabel('PC1', fontsize=11)
axes[0].set_ylabel('PC2', fontsize=11)

# Plot t-SNE
axes[1].scatter(Y_tsne_2d[:, 0], Y_tsne_2d[:, 1], c=y, cmap='tab10', alpha=0.5, s=10)
axes[1].set_title(f"t-SNE (Local Neighborhoods) | Time: {t_tsne:.2f}s", fontsize=14, fontweight='bold')
axes[1].set_xlabel('t-SNE 1', fontsize=11)
axes[1].set_ylabel('t-SNE 2', fontsize=11)

# Plot UMAP
axes[2].scatter(Y_lib[:, 0], Y_lib[:, 1], c=y, cmap='tab10', alpha=0.5, s=10)
axes[2].set_title(f"UMAP (Manifold Topology) | Time: {t_end - t_start:.2f}s", fontsize=14, fontweight='bold')
axes[2].set_xlabel('UMAP 1', fontsize=11)
axes[2].set_ylabel('UMAP 2', fontsize=11)

plt.suptitle('Projections Side-by-Side Comparison', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Hyperparameter Sweeps: Influence of `n_neighbors` and `min_dist`

### WHY?
UMAP's output topology is determined by its hyperparameters. We will run parameter sweeps over `n_neighbors` and `min_dist` to inspect how neighborhood scale and layout compactness affect cluster separation.

### HOW?
We loop through defined combinations of `n_neighbors` and `min_dist`, instantiate `umap.UMAP` for each configuration, and plot the resulting grid layouts side-by-side.

In [ ]:
n_neighbors_values = [5, 30]
min_dist_values = [0.01, 0.5]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for r_idx, n_neigh in enumerate(n_neighbors_values):
    for c_idx, m_dist in enumerate(min_dist_values):
        print(f"⚙️  Running UMAP with n_neighbors={n_neigh}, min_dist={m_dist}...")
        sweep_reducer = umap.UMAP(
            n_neighbors=n_neigh,
            min_dist=m_dist,
            n_components=2,
            random_state=42
        )
        Y_sweep = sweep_reducer.fit_transform(X_pca)
        
        ax = axes[r_idx, c_idx]
        ax.scatter(Y_sweep[:, 0], Y_sweep[:, 1], c=y, cmap='tab10', alpha=0.5, s=8)
        ax.set_title(f"n_neighbors: {n_neigh} | min_dist: {m_dist}", fontsize=12, fontweight='bold')
        ax.set_xlabel('UMAP 1', fontsize=9)
        ax.set_ylabel('UMAP 2', fontsize=9)

plt.suptitle('Hyperparameter Sweep Grid Dashboard', fontsize=16, fontweight='bold', y=0.95)
plt.show()

## 12. Evaluating UMAP Quality: Trustworthiness and KNN Classifiers

### WHY?
Unlike classification models, unsupervised projections have no direct ground-truth validation labels. We quantitatively evaluate UMAP projections using:
1. **Trustworthiness Score:** Measures the preservation of original neighbor structures (1.0 represents a perfect projection).
2. **KNN Classifier Accuracy:** Evaluates cluster classification accuracy on the 2D coordinates using cross-validation.

### HOW?
We calculate the `trustworthiness` score on a subset of the data, fit a 5-fold cross-validated KNN classifier on our 2D UMAP features, and plot the confusion matrix.

In [ ]:
print("📊 Evaluating UMAP Embedding Quality...\n")

# Step 12a: Calculate Trustworthiness Score (computed on 1000 samples for speed)
score = trustworthiness(X_pca[:1000], Y_lib[:1000], n_neighbors=10)
print(f"✅ Neighborhood Trustworthiness Score (n_neighbors=10): {score:.4f}")

# Step 12b: Train a KNN classifier on the 2D UMAP features
knn = KNeighborsClassifier(n_neighbors=5)
cv_scores = cross_val_score(knn, Y_lib, y, cv=5, scoring='accuracy')

print(f"✅ KNN (k=5) Cross-Validation Accuracy on 2D Embeddings: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

# Step 12c: Fit KNN on split data to plot a Confusion Matrix
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(Y_lib, y, test_size=0.3, random_state=42, stratify=y)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.arange(10), yticklabels=np.arange(10))
plt.title('Confusion Matrix: KNN Classifier on UMAP Embeddings', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.show()

# Placement & Interview Q&A

**Q1. What is the main mathematical difference between UMAP and t-SNE?**  
**Answer:** The primary differences are UMAP's cross-entropy loss function and local density scaling. t-SNE uses KL Divergence, which only penalizes points close in high-D but far in 2D, ignoring long-distance relationships. UMAP's cross-entropy contains a second term that acts as a repulsive force, preserving global topological configurations. Additionally, UMAP normalizes distances locally using adaptive bandwidth scales, avoiding t-SNE's expensive global denominator calculation.

**Q2. What is the role of the local connectivity parameter $\rho$ in UMAP?**  
**Answer:** $\rho_i$ is the distance from point $x_i$ to its single nearest neighbor. Subtracting $\rho_i$ from raw distances ensures that the similarity weight to the closest neighbor is exactly $\exp(0) = 1.0$. This mathematically guarantees that every point has a connection strength of $1.0$ to the manifold, preventing isolated outliers from being disconnected from the graph.

**Q3. How does UMAP achieve faster execution times than t-SNE?**  
**Answer:** UMAP speeds up nearest-neighbor search to $O(N \log N)$ using **NN-Descent** (projecting random projections and local sharing). During layout optimization, UMAP uses **Negative Sampling** instead of evaluating all pairs, keeping SGD execution times close to linear.

**Q4. Why does UMAP typically start with Spectral Embedding initialization?**  
**Answer:** Spectral embedding (Laplacian Eigenmaps) places the initial coordinates based on the eigenvalues/eigenvectors of the graph Laplacian matrix. Starting with a structured global layout rather than random coordinates speeds up Stochastic Gradient Descent convergence and ensures run-to-run consistency.

**Q5. Can UMAP transform new, unseen data points?**  
**Answer:** Yes. Because UMAP learns a topological representation of the training data, new data points can be mapped to the existing layout by calculating their similarities to the nearest neighbors in the training set and optimizing their coordinates accordingly, without modifying the training coordinates.

---

# Key Takeaways

- **UMAP balances local and global topology.** The cross-entropy loss function ensures attractive forces handle local neighborhoods while repulsive forces preserve global structures.
- **Spectral initialization improves convergence.** Starting with eigenvalues rather than random noise yields consistent, reproducible coordinates.
- **Hyperparameters require tuning.** Tuning `n_neighbors` adjusts the local-vs-global focus, while `min_dist` adjusts cluster compactness.
- **UMAP scales well to large datasets.** NN-Descent approximate nearest neighbors and negative sampling ensure near-linear time scaling.